In [1]:
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#Genere un archivo txt con datos relevantes. Abro el archivo, tomo los datos necesarios y luego cierro el archivo
archivo = open('Base de datos.txt', encoding='utf-8')
info = [dato.strip() for dato in archivo]
archivo.close()

In [4]:
#url de la API
api_url='https://api.yelp.com/v3/businesses/search'

#estos datos corresponden a una cuenta de usuario creada previamente
clientid = info[0]
apikey = info[1]
headers={'Authorization':'Bearer %s'%apikey}

In [5]:
ciudad = "Miami" # Se puede elegir cualquier ciudad del dataset entregado

In [6]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada. Voy guardando en "resultados" las 6 vueltas de 40 datos
resultados = []
for i in range(0,240,40):
    params={'term':'restaurants','location': ciudad,'limit':40, 'offset':i}
    response=requests.get(api_url,params=params,headers=headers)
    data=response.json()
    resultados.extend(data['businesses'])
    print(len(resultados))
    print(data)

40
{'businesses': [{'id': 'UXHxLN3DcDGI57uDIfCuJA', 'alias': 'olds-havana-cuban-bar-and-cocina-miami', 'name': "Old's Havana Cuban Bar & Cocina", 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg', 'is_closed': False, 'url': 'https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug', 'review_count': 3300, 'categories': [{'alias': 'cuban', 'title': 'Cuban'}, {'alias': 'bars', 'title': 'Bars'}, {'alias': 'venues', 'title': 'Venues & Event Spaces'}, {'alias': 'cuban', 'title': 'Cuban'}, {'alias': 'bars', 'title': 'Bars'}, {'alias': 'venues', 'title': 'Venues & Event Spaces'}], 'rating': 4.4, 'coordinates': {'latitude': 25.765607074496565, 'longitude': -80.21921771559332}, 'transactions': ['pickup'], 'price': '$$', 'location': {'address1': '1442 SW 8th St', 'address2': '', 'address3': '', 'city': 'Miami', 'zip_code': '33

In [7]:
#json_normalize abre los diccionarios anidados en columnas separadas (location.city, coordinates.latitude, etc.), y cada restaurante queda en su propia fila.
df_yelp = pd.json_normalize(resultados)
df_yelp.shape            # (240, n_columnas)

(240, 24)

In [8]:
#Para dejar la columna categories legible (viene como lista de diccionarios):
df_yelp['categorias'] = df_yelp['categories'].apply(lambda cats: ', '.join(c['title'] for c in cats))

In [9]:
#Para quedarte solo con las columnas más útiles:
df_yelp = df_yelp[['id', 'name', 'rating', 'review_count', 'price', 'categorias',
                   'location.city', 'location.zip_code', 'coordinates.latitude',
                   'coordinates.longitude', 'display_phone']]
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3300,$$,"Cuban, Bars, Venues & Event Spaces, Cuban, Bar...",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,4.8,183,NaN,"Sushi Bars, Asian Fusion, Wine Bars",Miami,33150,25.832632,-80.200107,(305) 788-0137
3,K3ukx2e11xTRtYBU01dmrA,Salty Flame,4.2,283,NaN,"Asian Fusion, Steakhouses",Miami,33131,25.760220,-80.192670,(305) 563-8972
4,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
5,_tiVacKyfWZjOljFenLwVQ,Giselle Miami,4.6,534,NaN,"Mediterranean, Lounges, Asian Fusion",Miami,33132,25.784964,-80.193414,(305) 358-9848
6,OJeHdB5l0Is3BNmYnpsd6w,Pho Nam,4.6,12,NaN,"Vietnamese, Vietnamese",Miami,33131,25.774042,-80.188561,(561) 519-9262
7,QQnw1bFxLyamuxn4giIy9g,Sufrat Mediterranean Grill,4.4,175,$$,"Mediterranean, Greek, Lebanese, Mediterranean,...",Doral,33166,25.820434,-80.335699,(305) 798-7278
8,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
9,eVhl0Yu1ZRWRD2D1OQGsYg,Capriccio,4.5,193,NaN,"Italian, Pizza, Steakhouses",Miami,33126,25.781633,-80.287662,(786) 936-5470


In [10]:
#Chequeos recomendados:
df_yelp['id'].duplicated().sum()   # Yelp a veces repite negocios entre páginas
df_yelp = df_yelp.drop_duplicates(subset='id').reset_index(drop=True)
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3300,$$,"Cuban, Bars, Venues & Event Spaces, Cuban, Bar...",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,4.8,183,NaN,"Sushi Bars, Asian Fusion, Wine Bars",Miami,33150,25.832632,-80.200107,(305) 788-0137
3,K3ukx2e11xTRtYBU01dmrA,Salty Flame,4.2,283,NaN,"Asian Fusion, Steakhouses",Miami,33131,25.760220,-80.192670,(305) 563-8972
4,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
5,_tiVacKyfWZjOljFenLwVQ,Giselle Miami,4.6,534,NaN,"Mediterranean, Lounges, Asian Fusion",Miami,33132,25.784964,-80.193414,(305) 358-9848
6,OJeHdB5l0Is3BNmYnpsd6w,Pho Nam,4.6,12,NaN,"Vietnamese, Vietnamese",Miami,33131,25.774042,-80.188561,(561) 519-9262
7,QQnw1bFxLyamuxn4giIy9g,Sufrat Mediterranean Grill,4.4,175,$$,"Mediterranean, Greek, Lebanese, Mediterranean,...",Doral,33166,25.820434,-80.335699,(305) 798-7278
8,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
9,eVhl0Yu1ZRWRD2D1OQGsYg,Capriccio,4.5,193,NaN,"Italian, Pizza, Steakhouses",Miami,33126,25.781633,-80.287662,(786) 936-5470


In [11]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [12]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 5900 restaurants


In [13]:
data['businesses'] # Hasta acá se le entregaría a los estudiantes

[{'id': 'gWeoN0eDy-tt05a9503jZg',
  'alias': 'hometown-bar-b-que-miami-2',
  'name': 'Hometown Bar-B-Que',
  'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/aLG1wp4ndd85N-s8tCFukw/o.jpg',
  'is_closed': False,
  'url': 'https://www.yelp.com/biz/hometown-bar-b-que-miami-2?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
  'review_count': 723,
  'categories': [{'alias': 'bbq', 'title': 'Barbeque'},
   {'alias': 'beerbar', 'title': 'Beer Bar'},
   {'alias': 'cocktailbars', 'title': 'Cocktail Bars'},
   {'alias': 'bbq', 'title': 'Barbeque'},
   {'alias': 'beerbar', 'title': 'Beer Bar'},
   {'alias': 'cocktailbars', 'title': 'Cocktail Bars'},
   {'alias': 'bbq', 'title': 'Barbeque'},
   {'alias': 'beerbar', 'title': 'Beer Bar'},
   {'alias': 'cocktailbars', 'title': 'Cocktail Bars'},
   {'alias': 'bbq', 'title': 'Barbeque'},
   {'alias': 'beerbar', 'title': 'Beer Bar'},
   {'alias': 'cocktailbars', 't

In [14]:
#Las filas que no tienen precio decido borrarlas
df_yelp = df_yelp.dropna(subset=['price']).reset_index(drop=True)
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3300,$$,"Cuban, Bars, Venues & Event Spaces, Cuban, Bar...",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
3,QQnw1bFxLyamuxn4giIy9g,Sufrat Mediterranean Grill,4.4,175,$$,"Mediterranean, Greek, Lebanese, Mediterranean,...",Doral,33166,25.820434,-80.335699,(305) 798-7278
4,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
5,hZm7TunlrksQbgS0ssXbUg,Versailles,4.0,7100,$$,"Cuban, Coffee & Tea, Sandwiches",Miami,33135,25.765390,-80.252874,(305) 444-0240
6,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,4.2,2923,$$$,"Mediterranean, Italian, Spanish",Miami,33131,25.762015,-80.188986,(305) 377-4442
7,qnCsyyEK01rRdrZgqqSfPA,TastiGo Kitchen,4.4,81,$$,"Chinese, Noodles",Miami,33133,25.740948,-80.238151,(305) 530-8315
8,Dzk_O70dBXoKcIQQeaOqyw,Amor di Pasta,4.3,427,$$,Italian,Miami,33126,25.776908,-80.288655,(786) 615-2019
9,nwY3CmIH75cN85cWBSaOAw,Apocalypse BBQ,4.4,1282,$$,"Food Trucks, Barbeque, Empanadas, Food Trucks,...",Miami,33183,25.687970,-80.393600,


In [15]:
#Queda un único DataFrame, df_seleccion, con los restaurantes que cumplen el criterio de cada nivel de precio. El reset_index(drop=True) renumera las filas desde 0.
# Grupo 1: $$ con rating > 4.4 y reviews > 974
top_dos = df_yelp[(df_yelp['price'] == '$$') &
                  (df_yelp['rating'] > 4.4) &
                  (df_yelp['review_count'] > 974)]

# Grupo 2: $$$ y $$$$ con rating > 4.3 y reviews > 427
top_caros = df_yelp[(df_yelp['price'].isin(['$$$', '$$$$'])) &
                    (df_yelp['rating'] > 4.3) &
                    (df_yelp['review_count'] > 427)]

# Unir en un solo DataFrame
df_seleccion = pd.concat([top_dos, top_caros]).reset_index(drop=True)

df_seleccion

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,9h9nMMvnINJWMt7TjJtePw,The Taco Stand,4.5,1764,$$,"Mexican, Mexican",Miami,33127,25.800782,-80.201316,(786) 580-4948
1,V3btCvWa1flTBYt2YKea_g,Bombay Darbar,4.5,1723,$$,"Indian, Indian",Miami,33133,25.729350,-80.240789,(305) 444-7272
2,fMk82DdXMJ_MULphyQM74w,La Fresa Francesa,4.6,1051,$$,"Wine Bars, French, Wine Bars, French, Wine Bar...",Hialeah,33010,25.824568,-80.282478,(786) 717-6886
3,nBIxqLEenuVRXQRTNiL3WA,Cajun Boil,4.7,1058,$$,"Cajun/Creole, Seafood",Miami,33174,25.768374,-80.360208,(786) 332-2641
4,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
5,1h8-r9cLeXxW99HhlRUnlA,CVI.CHE 105,4.4,6226,$$$,"Peruvian, Seafood",Miami,33132,25.775383,-80.188400,(305) 577-3454
6,M8dlhbSz62XnqJ4mP64L4w,Luca Osteria,4.5,510,$$$,Italian,Coral Gables,33134,25.751177,-80.257179,(305) 381-5097
7,Obt7Njta79-1CNtNwhNXiw,Bunbury,4.6,986,$$$,"Argentine, Steakhouses, Cocktail Bars",Miami,33132,25.788458,-80.193363,(305) 333-6929
8,MfkZSdJ0w4-SOLJiEyHU5A,Kyu,4.4,2133,$$$,Asian Fusion,Miami,33127,25.800890,-80.200170,(786) 577-0150
9,nTsEr_CQqsA8zwLV4kT5nA,Cafe La Trova,4.4,1338,$$$,"Cocktail Bars, Cuban, Venues & Event Spaces",Miami,33130,25.766266,-80.210630,(786) 615-4379
